### Synthetic Data Generation Examples

So, you want to either build out some realistic data to jumpstart your AI project or you need some for evals that doesn't violate the privacy of your customers/initial users/yourself. 

Let's evaluate and test out some of the potential approaches together. 

1. Simple and fully private: Use "dummy data" informed by your data types, language, basic distributions.

2. Generate data using a library with basic privacy guarantees.

3. Use a generative AI model +/or train your own with differential privacy. (Some links but no code in this notebook)

## Faker: fully synthetic and private 

[Faker](https://faker.readthedocs.io/en/master/) has been a trusted source of realistic test data for AI/ML but also software use. If you want to generate fully synthetic datasets that don't have any leakage from your actual data, I recommend starting here. 

There are also faker addons to generate things like CSVs, documents, etc. 

In [1]:
!pip install faker

In [2]:
from faker import Faker
from faker.providers import internet, python, phone_number

fake = Faker()

In [3]:
fake.name()

'Laura Smith'

In [4]:
fake.text()

'Box already fish again. Put improve word at challenge discuss drug. Sit provide anything religious final stop garden wide.\nSix nation high prevent. Avoid possible news rest put time fact.'

In [5]:
fake.add_provider(internet)

fake.ipv6()

'2201:d0da:5764:5842:e218:9894:b144:e1f0'

In [6]:
fake.add_provider(python)

fake.pyiterable()

('EOGqTTzKWaWwAvguQrHV',
 5068,
 'nrEUqdtuSTpusKWmomGO',
 -82731824708.271,
 8885,
 Decimal('205369829577144454881437935189756877769181687629421290475987872908026070214097522292941039009.4644258629786123591070194341558444460241938263109055043816127761606757137'),
 2389)

In [7]:
de_fake = Faker('de_DE')
for _ in range(10):
    print(de_fake.name())

Ioannis Krause
Maik Margraf
Ana Textor
Herr Leander Werner
Hellmut Steinberg
Prof. Sigismund Zobel B.A.
Serge Köhler MBA.
Gertraut Dehmel
Brita Benthin
Prof. Sylvana Scholtz


In [8]:
de_fake.add_provider(phone_number)

de_fake.phone_number()

'09617 76281'

There are also many ways to leverage libraries like [numpy](https://numpy.org/), [scipy](https://scipy.org/) and [scikit-learn](https://scikit-learn.org/stable/modules/clustering.html) or [pymc3](https://pypi.org/project/pymc3/) to develop more intricate fake-data-factories. 

You might start simple, using constraints around things like max/min/mean/median to build out a small function that leverages numpy or scipy. Or to decide things like: select random uniform from these 5 categories. 

Once that's working you can use more nuanced approaches, like learning a distribution via a clustering or Bayesian model based on your data. Then you can "sample" from an artificially created representative distribution using what you learned. 

I explored some of these ideas in [my datafuzz repo](https://github.com/kjam/datafuzz/blob/master/datafuzz/generators/core.py) in case you want a few small snippets to get your ideas flowing. 

## Probably Private: Using Sampling and Small Models via SDV

If you aren't sure what your data looks like or if it can change significantly and you don't want to continuously update your data generation, I recommend looking into [Synthetic Data Vault](https://docs.sdv.dev/sdv), a project that grew out of MIT and into a larger open-source library and company for creating synthetic data. 

The methods used by SDV vary depending on what parts of the library you are using. If you are using highly sensitive data and have significant privacy leakage concerns, I'd recommend familiarizing yourself with the core methods and determining what fits exactly for your use case.

Let's take a look together at an instructive example from their documentation.


In [9]:
!pip install sdv

In [10]:
from sdv.datasets.demo import download_demo

real_data, metadata = download_demo(
    modality='single_table',
    dataset_name='fake_hotel_guests'
)

In [11]:
metadata

{
    "tables": {
        "fake_hotel_guests": {
            "columns": {
                "guest_email": {
                    "sdtype": "email",
                    "pii": true
                },
                "has_rewards": {
                    "sdtype": "boolean"
                },
                "room_type": {
                    "sdtype": "categorical"
                },
                "amenities_fee": {
                    "sdtype": "numerical",
                    "computer_representation": "Float"
                },
                "checkin_date": {
                    "sdtype": "datetime",
                    "datetime_format": "%d %b %Y"
                },
                "checkout_date": {
                    "sdtype": "datetime",
                    "datetime_format": "%d %b %Y"
                },
                "room_rate": {
                    "sdtype": "numerical",
                    "computer_representation": "Float"
                },
                "billing_a

In [12]:
real_data.head()

,guest_email,has_rewards,room_type,amenities_fee,checkin_date,checkout_date,room_rate,billing_address,credit_card_number
0,michaelsanders@shaw.net,False,BASIC,37.89,27 Dec 2020,29 Dec 2020,131.23,"49380 Rivers Street\nSpencerville, AK 68265",4075084747483975747
1,randy49@brown.biz,False,BASIC,24.37,30 Dec 2020,02 Jan 2021,114.43,"88394 Boyle Meadows\nConleyberg, TN 22063",180072822063468
2,webermelissa@neal.com,True,DELUXE,0.00,17 Sep 2020,18 Sep 2020,368.33,"0323 Lisa Station Apt. 208\nPort Thomas, LA 82585",38983476971380
3,gsims@terry.com,False,BASIC,NaN,28 Dec 2020,31 Dec 2020,115.61,"77 Massachusetts Ave\nCambridge, MA 02139",4969551998845740
4,misty33@smith.biz,False,BASIC,16.45,05 Apr 2020,NaN,122.41,"1234 Corporate Drive\nBoston, MA 02116",3558512986488983


In [13]:
from sdv.single_table import GaussianCopulaSynthesizer

synthesizer = GaussianCopulaSynthesizer(metadata)
synthesizer.fit(real_data)

What did you just do? You created a new synthetic data generator that uses a [Gaussian Copula](https://en.wikipedia.org/wiki/Copula_(statistics)) as the "model". 

> From wiki: A copulas is a multivariate cumulative distribution function for which the marginal probability distribution of each variable is uniform on the interval [0, 1]. Copulas are used to describe / model the dependence (inter-correlation) between random variables.

Okay, so translated for non-statisticians, this is going to look at relationships between particular columns of your dataset and use them to figure out how those columns might change similarly or differently. It does so by making some assumptions on the underlying data, which may or may not fit, but it's a good starting point for finding these correlations. 

In [14]:
synthetic_data = synthesizer.sample(num_rows=500)
synthetic_data.head()

,guest_email,has_rewards,room_type,amenities_fee,checkin_date,checkout_date,room_rate,billing_address,credit_card_number
0,dsullivan@example.net,False,BASIC,0.29,27 Mar 2020,09 Mar 2020,135.15,"90469 Karla Knolls Apt. 781\nSusanberg, CA 70033",5161033759518983
1,steven59@example.org,False,DELUXE,8.15,07 Sep 2020,25 Jun 2020,183.24,"6108 Carla Ports Apt. 116\nPort Evan, MI 71694",4133047413145475690
2,brandon15@example.net,False,BASIC,11.65,22 Mar 2020,01 Apr 2020,163.57,86709 Jeremy Manors Apt. 786\nPort Garychester...,4977328103788
3,humphreyjennifer@example.net,False,BASIC,48.12,04 Jun 2020,14 May 2020,127.75,"8906 Bobby Trail\nEast Sandra, NY 43986",3524946844839485
4,joshuabrown@example.net,False,DELUXE,11.07,08 Jan 2020,13 Jan 2020,180.12,"732 Dennis Lane\nPort Nicholasstad, DE 49786",4446905799576890978


In [15]:
from sdv.evaluation.single_table import run_diagnostic

diagnostic = run_diagnostic(
    real_data=real_data,
    synthetic_data=synthetic_data,
    metadata=metadata
)

Generating report ...

(1/2) Evaluating Data Validity: |██████████████████████████████████████████████████████████████████████████████████████████████████████████████| 9/9 [00:00<00:00, 1676.23it/s]|
Data Validity Score: 100.0%

(2/2) Evaluating Data Structure: |██████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 487.82it/s]|
Data Structure Score: 100.0%

Overall Score (Average): 100.0%



Okay, what does that actually mean? 

From the [documentation](https://docs.sdv.dev/sdv/single-table-data/evaluation/diagnostic)

Validity checks:
  - Primary keys must always be unique and non-null
  - Continuous values in the synthetic data must adhere to the min/max range in the real data 
  - Discrete values in the synthetic data must adhere to the same categories as the real data

Structure checks to ensure the real and synthetic data have the same column names

In [16]:
from sdv.evaluation.single_table import evaluate_quality

quality_report = evaluate_quality(
    real_data,
    synthetic_data,
    metadata
)

Generating report ...

(1/2) Evaluating Column Shapes: |███████████████████████████████████████████████████████████████████████████████████████████████████████████████| 9/9 [00:00<00:00, 753.39it/s]|
Column Shapes Score: 90.06%

(2/2) Evaluating Column Pair Trends: |████████████████████████████████████████████████████████████████████████████████████████████████████████| 36/36 [00:00<00:00, 888.10it/s]|
Column Pair Trends Score: 83.47%

Overall Score (Average): 86.76%



Okay, what does that actually mean? 

From the [documentation](https://docs.sdv.dev/sdv/single-table-data/evaluation/data-quality)

- Column Shapes: The statistical similarity between the real and synthetic data for single columns of data. This is often called the marginal distribution of each column.

So this is comparing column A in your raw data directly with column A in the synthetic data and looking at the overall distribution of the column. (i.e. do all the values in that column look relatively similar in their amount and repetition?)

- Column Pair Trends: The statistical similarity between the real and synthetic data for pairs of columns. This is often called the correlation or bivariate distributions of the columns.    

This is where it's getting a bit more interesting. This is holding Column A with Column B and looking at if Column A has a particular value, does the value correlate with column B in a similar way between the real and synthetic data. For example, this could look at if your sales tax rate matches similarly with the location. 

In [17]:
quality_report.get_details('Column Shapes')

,Column,Metric,Score
0,has_rewards,TVComplement,0.982000
1,room_type,TVComplement,0.984000
2,amenities_fee,KSComplement,0.764778
3,checkin_date,KSComplement,0.962000
4,checkout_date,KSComplement,0.968750
5,room_rate,KSComplement,0.742000


You can read more about the calculation of the [TVComplement](https://docs.sdv.dev/sdmetrics/data-metrics/quality/tvcomplement) which is used for categorical columns and the [KSComplement](https://docs.sdv.dev/sdmetrics/data-metrics/quality/kscomplement) for continuous value (i.e. numbers, floats, values that are not a particular categorization).

In [18]:
quality_report.get_details('Column Pair Trends')

,Column 1,Column 2,Metric,Score,Real Correlation,Synthetic Correlation,Real Association,Meets Threshold?
0,has_rewards,room_type,ContingencySimilarity,0.940000,NaN,NaN,0.374138,True
1,has_rewards,amenities_fee,ContingencySimilarity,0.654000,NaN,NaN,0.882993,True
2,has_rewards,checkin_date,ContingencySimilarity,NaN,NaN,NaN,0.214637,False
3,has_rewards,checkout_date,ContingencySimilarity,NaN,NaN,NaN,0.227684,False
4,has_rewards,room_rate,ContingencySimilarity,0.856000,NaN,NaN,0.307499,True
5,room_type,amenities_fee,ContingencySimilarity,NaN,NaN,NaN,0.249106,False
6,room_type,checkin_date,ContingencySimilarity,NaN,NaN,NaN,0.185876,False
7,room_type,checkout_date,ContingencySimilarity,NaN,NaN,NaN,0.184554,False
8,room_type,room_rate,ContingencySimilarity,0.738000,NaN,NaN,0.683544,True
9,amenities_fee,checkin_date,CorrelationSimilarity,NaN,0.031410,NaN,NaN,False


Ooo, cool! So now we see a bit more on the expected correlations and how they are rated based on the [correlation similarity](https://docs.sdv.dev/sdmetrics/data-metrics/quality/correlationsimilarity). Here we can start to notice where our synthetic data is diverging from the real correlations.

In [21]:
from sdv.evaluation.single_table import get_column_plot

fig = get_column_plot(
    real_data=real_data,
    synthetic_data=synthetic_data,
    column_name='amenities_fee',
    metadata=metadata
)

fig.show()

In [22]:
from sdv.evaluation.single_table import get_column_pair_plot

fig = get_column_pair_plot(
    real_data=real_data,
    synthetic_data=synthetic_data,
    column_names=['room_rate', 'room_type'],
    metadata=metadata
)

fig.show()

In [34]:
custom_synthesizer = GaussianCopulaSynthesizer(
    metadata,
    default_distribution='truncnorm',
    numerical_distributions={
        'checkin_date': 'uniform',
        'checkout_date': 'uniform',
        'amenities_fee': 'gamma',
        'room_rate': 'gaussian_kde'
    }
)

custom_synthesizer.fit(real_data)

Okay, so now you're being a bit more specific about how you want the distributions to be fit. For example, you are asking for a uniform distribution for the check-in and check-out and using a [Gaussian Kernel Density Estimator](https://en.wikipedia.org/wiki/Kernel_density_estimation) for room rates and a [Gamma distribution](https://en.wikipedia.org/wiki/Gamma_distribution) for the amenities fees. The default for the others uses a truncated normal distribution. 

This will probably help with the slight skew that you saw in the original synthetic data generation. You can check out other ways to modify the distribution choices [in the Gaussian Copula Synthesizer documentation](https://docs.sdv.dev/sdv/single-table-data/modeling/synthesizers/gaussiancopulasynthesizer).

In [39]:
learned_distributions = custom_synthesizer.get_learned_distributions()
learned_distributions['has_rewards']

{'distribution': 'truncnorm',
 'learned_parameters': {'a': np.float64(-0.5415003201568632),
  'b': np.float64(0.4646073705121775),
  'loc': np.float64(0.5390634685759674),
  'scale': np.float64(0.9878956258627398)}}

In [40]:
synthetic_data_customized = custom_synthesizer.sample(num_rows=500)

quality_report = evaluate_quality(
    real_data,
    synthetic_data_customized,
    metadata
)

Generating report ...

(1/2) Evaluating Column Shapes: |██████████████████████████████████████████████████████████████████████████████████████████████████████████████| 9/9 [00:00<00:00, 2177.48it/s]|
Column Shapes Score: 95.22%

(2/2) Evaluating Column Pair Trends: |███████████████████████████████████████████████████████████████████████████████████████████████████████| 36/36 [00:00<00:00, 1604.50it/s]|
Column Pair Trends Score: 88.76%

Overall Score (Average): 91.99%



In [41]:
fig = get_column_plot(
    real_data=real_data,
    synthetic_data=synthetic_data_customized,
    column_name='room_rate',
    metadata=metadata
)

fig.show()

In [42]:
fig = get_column_plot(
    real_data=real_data,
    synthetic_data=synthetic_data_customized,
    column_name='amenities_fee',
    metadata=metadata
)

fig.show()

Now you have data that looks a bit more like your real data, perhaps helping you better test your AI workflows and make realistic evaluation data!

You might want to check out some other ways to generate data with SDV such as the [Contstraint Augmented Generator](https://docs.sdv.dev/sdv/concepts/constraint-augmented-generation-cag), which is useful if you have specific business relationships and rules, or the [HMASynthesizer](https://docs.sdv.dev/sdv/multi-table-data/modeling/synthesizers/hmasynthesizer), which is similar to the one you just used but for multiple tables. 

In [43]:
# Final TODO: add links for deep learning :) 

### Differentially Private Deep Learning for Synthetic Data 

Since you know that [deep learning models can memorize their training data](https://blog.kjamistan.com/a-deep-dive-into-memorization-in-deep-learning.html), you'll probably be interested in how to safely produce synthetic data via today's generative AI systems.

One way is to simply know of this risk and to try to carefully use and review outputs from generative models without adding too much business context. You can check out Notebook 2 in this repo for a small example. 

But if you get deeper into modeling your data, you'll want not only higher quality synthetic data, but also ones that are better informed by your operating context and real-world use cases. This means making decisions like:

- Are we going to use a third-party synthetic learning platform and train with our real data knowing that there might be privacy leakage?
- Are we equipped to train our own generative models either with open-weight models and fine tuning or training our own models?
- If so, what privacy testing and evaluations can we run to ensure we understand the privacy leakage?

In doing so, you'll likely want to learn about:


- [Training and Fine-Tuning Deep Learning with Differential Privacy](https://blog.kjamistan.com/differential-privacy-in-deep-learning.html#differential-privacy-in-deep-learning)
- [Running Accounting and Auditing for your training/fine-tuning](https://blog.kjamistan.com/differential-privacy-parameters-accounting-and-auditing-in-deep-learning-and-ai.html)
- [PyTorch's Opacus](https://opacus.ai/)
- [Privacy Attacks to run for testing and evaluating](https://blog.kjamistan.com/defining-privacy-attacks-in-ai-and-ml.html)

One of my favorite libraries to recommend on training your  has since been acquired and I'm not sure what the longer term plan is from NVIDIA, but here are [their public archived repos](https://github.com/gretelai) for your perusal.